In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# تهيئة إدارة الضوضاء مع Estimator

<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
هناك عدة طرق لإدارة الضوضاء، وعادةً ما يتم ذلك باستخدام تقنيات متنوعة لتخفيف الأخطاء وتثبيطها لتفادي الأخطاء قبل حدوثها. تُسبِّب هذه التقنيات عادةً حملاً إضافيًا في المعالجة المسبقة. لذلك، من المهم تحقيق توازن بين تحسين النتائج والتأكد من إتمام مهمتك في وقت معقول.

يدعم Estimator تقنيات إدارة الضوضاء التالية. راجع [تقنيات تخفيف الأخطاء وتثبيطها](error-mitigation-and-suppression-techniques) للاطلاع على شرح لكل منها. راجع قسم [إعدادات الأخطاء المخصصة](#advanced-error) للحصول على تعليمات تفعيل هذه التقنيات.

- [الفصل الديناميكي](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [تدوير Pauli](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## مستوى المرونة
يُحدِّد `resilience_level` مقدار المرونة المطلوب بناؤها في مواجهة الأخطاء. تُولِّد المستويات الأعلى نتائج أكثر دقة، على حساب أوقات معالجة أطول. يمكن استخدام مستويات المرونة لضبط توازن التكلفة/الدقة عند تطبيق إدارة الضوضاء على استعلام الـ primitive. تُقلِّل إدارة الضوضاء الأخطاء (الانحياز) في النتائج عبر معالجة المخرجات من مجموعة أو طيف من الـ Circuits المترابطة. تعتمد درجة تقليل الأخطاء على الطريقة المُطبَّقة. يُجرِّد مستوى المرونة الاختيار التفصيلي لطريقة إدارة الضوضاء للسماح للمستخدمين بالتفكير في توازن التكلفة/الدقة المناسب لتطبيقهم.

بناءً على ذلك، يتوافق كل مستوى مع طريقة أو طرق ذات مستوى متزايد من الحمل الزائد في أخذ العينات الكمية لتمكينك من تجربة مقايضات مختلفة بين الوقت والدقة. يوضح الجدول التالي المستويات والطرق المقابلة المتاحة لكل من الـ primitives.

<span id="resilience-table"></span>

| مستوى المرونة | الوصف | التقنية |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0 | بدون تخفيف | لا شيء |
| 1 [افتراضي] | الحد الأدنى من تكاليف التخفيف: تخفيف الأخطاء المرتبطة بأخطاء القراءة | تدوير قياس [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex) |
| 2 | تكاليف تخفيف متوسطة. يُقلِّل عادةً من الانحياز في المُقدِّرات، لكن غير مضمون أن يكون خالٍ من الانحياز. | المستوى 1 + [الاستقراء عند تعامد الضوضاء (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) وتدوير البوابات

> **Info:** مستويات المرونة في مرحلة بيتا حاليًا لذا قد يتفاوت الحمل الزائد في أخذ العينات وجودة الحل من Circuit إلى أخرى. ستُطلَق ميزات جديدة وخيارات متقدمة وأدوات إدارة بشكل تدريجي. لا يُضمَن تطبيق طرق محددة لإدارة الضوضاء عند كل مستوى مرونة.

### تهيئة Estimator بمستويات المرونة
يمكنك استخدام مستويات المرونة لتحديد تقنيات إدارة الضوضاء، أو يمكنك ضبط تقنيات مخصصة بشكل فردي كما هو موضح في [إعدادات الأخطاء المخصصة](#advanced-error).

> **Note:** أي خيارات تُحددها يدويًا إضافةً إلى مستوى المرونة تُطبَّق بالإضافة إلى مجموعة الخيارات الأساسية المُعرَّفة بواسطة مستوى المرونة. لذلك، من حيث المبدأ، يمكنك ضبط مستوى المرونة على 1، لكن بعد ذلك إيقاف تخفيف القياس، وإن لم يكن ذلك مُنصَحًا.
> 
> على سبيل المثال، ضبط مستوى المرونة على 0 يوقف `zne_mitigation`، لكن `estimator.options.resilience.zne_mitigation = True` يُلغي تلك القيمة.

### مثال
الكود التالي يُفعِّل ZNE وTREX وتدوير البوابات عبر ضبط `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## إعدادات إدارة الضوضاء المخصصة
يمكنك تشغيل وإيقاف طرق إدارة الضوضاء الفردية باستخدام [خيارات Estimator](/guides/estimator-options).

> **Note:** لا تعمل جميع الخيارات معًا على جميع أنواع الـ Circuits. راجع [جدول توافق الميزات](estimator-options#options-compatibility-table) للاطلاع على التفاصيل.

### مثال

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>